# Notebook 2: Frozen Embedding Baseline

Level 1 approach: use ESM2 and ProtT5 as **frozen** feature extractors,
then train XGBoost, LightGBM, and MLP classifiers on top.

This establishes a strong baseline without any fine-tuning.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import pickle
import torch
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModel, T5EncoderModel
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.multioutput import MultiOutputClassifier
import xgboost as xgb
import lightgbm as lgb

from preprocessing import load_processed
from evaluate import compute_fmax, compute_aupr, evaluate_ontology
from dataset import ESM2Dataset, ProtT5Dataset, collate_fn_pad

sns.set_style('whitegrid')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Load Preprocessed Data

In [ ]:
# Load BPO split (run preprocessing.py first)
data = load_processed('../data/processed/cafa6_bpo.pkl')
go_terms = data['go_terms']
print(f'GO terms: {len(go_terms):,}')
print(f'Train: {len(data["train"]["protein_ids"]):,} | Val: {len(data["val"]["protein_ids"]):,} | Test: {len(data["test"]["protein_ids"]):,}')

## 2. Extract ESM2 Embeddings

In [ ]:
ESM2_MODEL = 'facebook/esm2_t6_8M_UR50D'  # 8M param — fast for baseline

tokenizer_esm2 = AutoTokenizer.from_pretrained(ESM2_MODEL)
backbone_esm2 = AutoModel.from_pretrained(ESM2_MODEL).to(DEVICE).eval()

@torch.no_grad()
def extract_esm2_embeddings(sequences, batch_size=32, max_length=512):
    dataset = ESM2Dataset(sequences, tokenizer_esm2, max_length)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_pad)
    embeddings = []
    for batch in tqdm(loader, desc='ESM2 embedding'):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        out = backbone_esm2(input_ids=input_ids, attention_mask=attention_mask)
        # Mean pool
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        embeddings.append(pooled.cpu().numpy())
    return np.concatenate(embeddings, axis=0)

X_train_esm2 = extract_esm2_embeddings(data['train']['sequences'])
X_val_esm2   = extract_esm2_embeddings(data['val']['sequences'])
X_test_esm2  = extract_esm2_embeddings(data['test']['sequences'])
print(f'ESM2 embedding shape: {X_train_esm2.shape}')

## 3. Extract ProtT5 Embeddings

In [ ]:
PROTT5_MODEL = 'Rostlab/prot_t5_xl_uniref50'

tokenizer_t5 = AutoTokenizer.from_pretrained(PROTT5_MODEL)
backbone_t5 = T5EncoderModel.from_pretrained(PROTT5_MODEL).to(DEVICE).eval()

@torch.no_grad()
def extract_prott5_embeddings(sequences, batch_size=8, max_length=512):
    dataset = ProtT5Dataset(sequences, tokenizer_t5, max_length)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_pad)
    embeddings = []
    for batch in tqdm(loader, desc='ProtT5 embedding'):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        out = backbone_t5(input_ids=input_ids, attention_mask=attention_mask)
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        embeddings.append(pooled.cpu().numpy())
    return np.concatenate(embeddings, axis=0)

X_train_t5 = extract_prott5_embeddings(data['train']['sequences'])
X_val_t5   = extract_prott5_embeddings(data['val']['sequences'])
X_test_t5  = extract_prott5_embeddings(data['test']['sequences'])
print(f'ProtT5 embedding shape: {X_train_t5.shape}')

## 4. Train Classifiers on ESM2 Embeddings

In [ ]:
Y_train = data['train']['labels']
Y_val   = data['val']['labels']
Y_test  = data['test']['labels']

# Limit to top-100 terms for demonstration (XGBoost is slow on thousands)
N_TERMS = min(100, len(go_terms))
Y_tr = Y_train[:, :N_TERMS]
Y_v  = Y_val[:, :N_TERMS]
Y_ts = Y_test[:, :N_TERMS]

results = {}

# --- Logistic Regression ---
lr = MultiOutputClassifier(LogisticRegression(max_iter=300, C=1.0), n_jobs=-1)
lr.fit(X_train_esm2, Y_tr)
lr_scores = np.column_stack([e.predict_proba(X_val_esm2)[:, 1] for e in lr.estimators_])
fmax_lr, _ = compute_fmax(Y_v, lr_scores)
aupr_lr = compute_aupr(Y_v, lr_scores)
results['LogReg (ESM2)'] = {'Fmax': fmax_lr, 'AUPR': aupr_lr}
print(f'LogReg ESM2 | Fmax={fmax_lr:.4f} | AUPR={aupr_lr:.4f}')

In [ ]:
# --- XGBoost ---
xgb_model = MultiOutputClassifier(
    xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                      use_label_encoder=False, eval_metric='logloss', n_jobs=-1),
    n_jobs=1,
)
xgb_model.fit(X_train_esm2, Y_tr)
xgb_scores = np.column_stack([e.predict_proba(X_val_esm2)[:, 1] for e in xgb_model.estimators_])
fmax_xgb, _ = compute_fmax(Y_v, xgb_scores)
aupr_xgb = compute_aupr(Y_v, xgb_scores)
results['XGBoost (ESM2)'] = {'Fmax': fmax_xgb, 'AUPR': aupr_xgb}
print(f'XGBoost ESM2 | Fmax={fmax_xgb:.4f} | AUPR={aupr_xgb:.4f}')

In [ ]:
# --- LightGBM ---
lgb_model = MultiOutputClassifier(
    lgb.LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, n_jobs=-1, verbose=-1),
    n_jobs=1,
)
lgb_model.fit(X_train_esm2, Y_tr)
lgb_scores = np.column_stack([e.predict_proba(X_val_esm2)[:, 1] for e in lgb_model.estimators_])
fmax_lgb, _ = compute_fmax(Y_v, lgb_scores)
aupr_lgb = compute_aupr(Y_v, lgb_scores)
results['LightGBM (ESM2)'] = {'Fmax': fmax_lgb, 'AUPR': aupr_lgb}
print(f'LightGBM ESM2 | Fmax={fmax_lgb:.4f} | AUPR={aupr_lgb:.4f}')

## 5. Results Table

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('Fmax', ascending=False)
print(results_df.to_string(float_format='{:.4f}'.format))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
results_df['Fmax'].plot.bar(ax=axes[0], color='steelblue', title='Fmax (Val)')
results_df['AUPR'].plot.bar(ax=axes[1], color='coral', title='AUPR (Val)')
for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../reports/baseline_results.png', dpi=150)
plt.show()